# Chapter 20: Concurrent Executors
*Fluent Python, 2nd Edition -- Luciano Ramalho*

This notebook distills the key ideas from Chapter 20 into runnable, self-contained examples.
All code targets **Python 3.11+** and uses only the standard library (no network calls needed).

**Concepts covered:**
1. The `concurrent.futures` package (ThreadPoolExecutor, ProcessPoolExecutor)
2. `executor.map` -- parallel map with ordered results
3. Future objects -- `submit`, `result`, `as_completed`
4. ProcessPoolExecutor for CPU-bound work
5. Ordering and blocking behavior of `executor.map`
6. Error handling patterns with `as_completed`

---
## 1. The `concurrent.futures` Package

The module provides two high-level executor classes that share the same interface:

| Class | Worker type | Best for |
|---|---|---|
| `ThreadPoolExecutor` | OS threads | I/O-bound tasks (HTTP, file I/O) |
| `ProcessPoolExecutor` | OS processes | CPU-bound tasks (bypasses GIL) |

Both implement the `Executor` abstract interface with two key methods:
- **`executor.map(fn, *iterables)`** -- parallel version of `map()`
- **`executor.submit(fn, *args)`** -- schedule a single callable, returns a `Future`

In [ ]:
from concurrent import futures
import os

# Quick check: what's the default max_workers for ThreadPoolExecutor?
# Since Python 3.8: min(32, os.cpu_count() + 4)
default_workers = min(32, (os.cpu_count() or 1) + 4)
print(f"CPU cores: {os.cpu_count()}")
print(f"Default ThreadPoolExecutor max_workers: {default_workers}")

---
## 2. ThreadPoolExecutor and `executor.map`

The simplest way to run tasks concurrently: pass a function and an iterable.
Results come back **in submission order**, just like the built-in `map()`.

Below we simulate I/O-bound work (sleeping) to see the speedup.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def fake_download(country_code: str) -> str:
    """Simulate downloading a flag image (I/O-bound)."""
    time.sleep(0.2)  # simulate network latency
    return f"{country_code}: OK"

codes = ['BR', 'CN', 'DE', 'EG', 'FR', 'ID', 'IN', 'JP', 'MX', 'US']

# --- Sequential baseline ---
t0 = time.perf_counter()
seq_results = [fake_download(cc) for cc in codes]
seq_elapsed = time.perf_counter() - t0
print(f"Sequential: {seq_elapsed:.2f}s  ({len(seq_results)} results)")

# --- Concurrent with ThreadPoolExecutor.map ---
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as executor:
    conc_results = list(executor.map(fake_download, codes))
conc_elapsed = time.perf_counter() - t0
print(f"Threaded:   {conc_elapsed:.2f}s  ({len(conc_results)} results)")
print(f"Speedup:    {seq_elapsed / conc_elapsed:.1f}x")

# Results arrive in SUBMISSION order
print(f"\nFirst 3 results: {conc_results[:3]}")

---
## 3. Future Objects: `submit`, `result`, `as_completed`

A **Future** represents a deferred computation.

Key facts:
- You never create Futures yourself -- the executor creates them via `submit()`.
- `future.done()` -- non-blocking check: has the callable finished?
- `future.result()` -- blocks until the result is ready (or raises the exception).
- `future.add_done_callback(fn)` -- register a callback for when the future completes.
- `futures.as_completed(fs)` -- yields futures **as they finish** (out of order).

### `executor.map` vs `executor.submit` + `as_completed`
| | `executor.map` | `submit` + `as_completed` |
|---|---|---|
| Result order | submission order | completion order |
| Flexibility | same callable | different callables per task |
| Error handling | exception on iteration | exception via `future.result()` |

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed, Future

def slow_task(name: str, seconds: float) -> str:
    """Simulate variable-duration work."""
    time.sleep(seconds)
    return f"{name} done ({seconds}s)"

tasks = [("fast-A", 0.1), ("slow-B", 0.5), ("medium-C", 0.3)]

with ThreadPoolExecutor(max_workers=3) as executor:
    # submit() returns a Future for each task
    future_to_name: dict[Future, str] = {}
    for name, secs in tasks:
        f = executor.submit(slow_task, name, secs)
        future_to_name[f] = name
        print(f"Submitted {name}: {f}")

    print("\n--- Results as they complete ---")
    for f in as_completed(future_to_name):
        name = future_to_name[f]
        result = f.result()  # won't block -- future is already done
        print(f"  {result}")

---
## 4. ProcessPoolExecutor for CPU-Bound Work

For CPU-intensive tasks, threads don't help because of the GIL.
`ProcessPoolExecutor` spawns separate OS processes, each with its own GIL.

Switching is trivial -- just change `ThreadPoolExecutor` to `ProcessPoolExecutor`.

> **Note:** `ProcessPoolExecutor` defaults `max_workers` to `os.cpu_count()`.

In [ ]:
import time
import math
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def is_prime(n: int) -> bool:
    """Simple primality test (CPU-bound)."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for i in range(3, math.isqrt(n) + 1, 2):
        if n % i == 0:
            return False
    return True

NUMBERS = [
    142702110479723,
    299593572317531,
    3333333333333301,
    3333333333333333,
    4444444444444423,
    4444444444444444,
    5555555555555503,
    5555555555555555,
]

def check_prime(n: int) -> tuple[int, bool, float]:
    t0 = time.perf_counter()
    result = is_prime(n)
    elapsed = time.perf_counter() - t0
    return (n, result, elapsed)

# --- Threaded (limited by GIL for CPU work) ---
t0 = time.perf_counter()
with ThreadPoolExecutor() as executor:
    thread_results = list(executor.map(check_prime, NUMBERS))
thread_time = time.perf_counter() - t0

# --- Process-based (true parallelism) ---
t0 = time.perf_counter()
with ProcessPoolExecutor() as executor:
    proc_results = list(executor.map(check_prime, NUMBERS))
proc_time = time.perf_counter() - t0

print(f"{'Number':>20}  Prime?  Elapsed")
print("-" * 45)
for n, prime, elapsed in proc_results:
    label = 'P' if prime else ' '
    print(f"{n:>20}    {label}    {elapsed:.4f}s")

print(f"\nThreadPoolExecutor total:  {thread_time:.2f}s")
print(f"ProcessPoolExecutor total: {proc_time:.2f}s")
cores = __import__('os').cpu_count()
print(f"(using {cores} CPU cores)")

---
## 5. `executor.map` Ordering and Blocking Behavior

A subtle but important property: `executor.map` returns a **generator** that yields
results in **submission order**. If the first task is slow, `next()` blocks even though
later tasks may have finished.

This example (adapted from the book's `demo_executor_map.py`) demonstrates the effect.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def loiter(n: int) -> int:
    """Sleep for n*0.1 seconds, return n*10."""
    tag = '\t' * n
    print(f"  {tag}loiter({n}): start (will take {n*0.1:.1f}s)")
    time.sleep(n * 0.1)
    print(f"  {tag}loiter({n}): done.")
    return n * 10

print("Submitting loiter(0..4) to a pool of 3 workers:\n")
executor = ThreadPoolExecutor(max_workers=3)
results = executor.map(loiter, range(5))
print(f"\nresults object: {results!r}\n")

print("Iterating over results (blocks on each in order):")
for i, result in enumerate(results):
    print(f"  >>> result {i}: {result}")

executor.shutdown(wait=True)

---
## 6. Error Handling and Progress with `as_completed`

The recommended pattern from the book:
1. Use `executor.submit()` to schedule each task.
2. Store `future -> context` in a dict.
3. Iterate with `as_completed()` -- call `future.result()` inside `try/except`.
4. On error, look up context from the dict for a useful error message.

This decouples **scheduling** from **result processing** and lets you report progress
as each task finishes (useful for progress bars like `tqdm`).

In [ ]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed, Future
from collections import Counter
from enum import Enum

class Status(Enum):
    OK = 'ok'
    NOT_FOUND = 'not_found'
    ERROR = 'error'

def flaky_download(cc: str) -> Status:
    """Simulate a download that sometimes fails."""
    time.sleep(random.uniform(0.05, 0.2))
    roll = random.random()
    if roll < 0.15:
        raise ConnectionError(f"Timeout fetching {cc}")
    if roll < 0.30:
        raise FileNotFoundError(f"{cc} not found on server")
    return Status.OK

random.seed(42)  # reproducible demo
countries = ['BR', 'CN', 'DE', 'EG', 'FR', 'ID', 'IN', 'JP', 'MX', 'NG',
             'PH', 'PK', 'RU', 'TR', 'US', 'VN']

counter: Counter[Status] = Counter()

with ThreadPoolExecutor(max_workers=5) as executor:
    # Step 1 & 2: submit and build future -> context dict
    future_to_cc: dict[Future, str] = {}
    for cc in countries:
        f = executor.submit(flaky_download, cc)
        future_to_cc[f] = cc

    # Step 3 & 4: process results as they complete
    for future in as_completed(future_to_cc):
        cc = future_to_cc[future]
        try:
            status = future.result()
        except FileNotFoundError:
            status = Status.NOT_FOUND
        except ConnectionError as exc:
            status = Status.ERROR
            print(f"  [{cc}] {exc}")
        counter[status] += 1

print(f"\n--- Download Report ---")
for s in Status:
    print(f"  {s.name:>10}: {counter[s]}")
print(f"  {'TOTAL':>10}: {sum(counter.values())}")

---
## Key Takeaways

1. **`concurrent.futures` hides complexity** -- threads, processes, and queues
   are managed for you behind a clean `Executor` interface.

2. **`executor.map`** is the simplest API -- parallel `map()` with results in
   submission order. One line replaces a sequential `for` loop.

3. **`executor.submit` + `as_completed`** gives more control:
   - Results arrive in *completion* order (faster feedback).
   - You can submit *different* callables.
   - You can handle errors per-task.

4. **Switching from threads to processes is trivial** -- just change the class name.
   Use `ThreadPoolExecutor` for I/O-bound work, `ProcessPoolExecutor` for CPU-bound.

5. **The `future -> context` dict pattern** is essential for real-world code:
   map each future to its context data at submission time, then look it up
   when the future completes for error reporting, logging, or follow-up processing.

---
*Notebook generated from Fluent Python Chapter 20 distillation.*